# Кандидатогенерация для поиска услуг Авито

Ноутбук содержит два этапа
1. Построение локальной валидации на train.parquet и замер метрики Recall@50
2. Финальный инференс на тестовых файлах и генерация answer.csv

In [1]:
import time
import numpy as np
import pandas as pd
import polars as pl
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'устройство: {device}')

d:\multi-stage search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


устройство: cuda


In [2]:
# функция расчета целевой метрики recall@k
def compute_recall_at_k(predictions, ground_truth, k=50):
    if not ground_truth:
        return 0.0
    recalls = []
    for q_id, target_items in ground_truth.items():
        if not target_items:
            continue
        pred_items = set(predictions.get(q_id, [])[:k])
        recalls.append(len(pred_items & set(target_items)) / len(target_items))
    return float(np.mean(recalls)) if recalls else 0.0

## 1. Локальная валидация на train.parquet

In [3]:
# загрузка train выборки
train_df = pd.read_parquet('dataset/train.parquet')
print(f'строк в train: {len(train_df)}')

# группировка уникальных поисковых контекстов
search_cols = ['search_query', 'search_location_id', 'search_category', 'search_infm_params_text']
train_df['query_uid'] = train_df.groupby(search_cols).ngroup()

строк в train: 497673


In [4]:
# выделение корпуса уникальных товаров через polars
item_cols = [c for c in train_df.columns if c.startswith('item_')]
train_corpus_df = (
    pl.scan_parquet('dataset/train.parquet')
    .select(item_cols)
    .unique(subset=['item_id'])
    .collect()
    .to_pandas()
)
print(f'уникальных товаров в корпусе train: {len(train_corpus_df)}')

уникальных товаров в корпусе train: 344825


In [5]:
# сборка валидационного сплита из 1545 редких и 907 частых запросов
np.random.seed(42)
query_counts = train_df['search_query'].value_counts()
rare_queries = query_counts[query_counts >= 1].index.values
cold_query_texts = set(np.random.choice(rare_queries, size=1545, replace=False))

multi_queries = query_counts[query_counts >= 3].index.values
warm_query_texts = set(np.random.choice(multi_queries, size=907, replace=False))

cold_all_uids = train_df[train_df['search_query'].isin(cold_query_texts)]['query_uid'].drop_duplicates()
cold_val_uids = cold_all_uids.sample(n=1545, random_state=42)

warm_rows_all = train_df[train_df['search_query'].isin(warm_query_texts)]
warm_all_uids = warm_rows_all['query_uid'].drop_duplicates()
warm_val_uids = warm_all_uids.sample(n=907, random_state=42)

val_selected_uids = set(pd.concat([cold_val_uids, warm_val_uids]))
val_df_honest = train_df[train_df['query_uid'].isin(val_selected_uids)].reset_index(drop=True)
val_ground_truth_honest = val_df_honest.groupby('query_uid')['item_id'].apply(set).to_dict()

query_cols_val = ['query_uid', 'search_query', 'search_location_id', 'search_category', 'search_infm_params_text', 'search_is_delivery_search']
val_queries_honest = val_df_honest[query_cols_val].drop_duplicates(subset=['query_uid']).reset_index(drop=True)
val_queries_honest['is_cold'] = val_queries_honest['search_query'].isin(cold_query_texts)

print(f'запросов в валидации: {len(val_queries_honest)}, холодных: {val_queries_honest["is_cold"].sum()}')

запросов в валидации: 2452, холодных: 1552


In [6]:
# текстовая подготовка корпуса и запросов валидации
pl_c = pl.from_pandas(train_corpus_df)
tc = pl_c['item_title_raw'].fill_null('') + ' '
pc = pl_c['item_infm_params_text'].fill_null('') + ' '
dc = pl_c['item_description_raw'].fill_null('')

train_corpus_df['corpus_text'] = (
    (tc + tc + tc + pc + pc + dc)
    .str.to_lowercase()
    .str.replace_all('ё', 'е')
    .to_pandas()
)

pl_vq = pl.from_pandas(val_queries_honest)
vq = pl_vq['search_query'].fill_null('') + ' '
vqp = pl_vq['search_infm_params_text'].fill_null('')

val_queries_honest['query_text'] = (
    (vq + vqp)
    .str.to_lowercase()
    .str.replace_all('ё', 'е')
    .to_pandas()
)

In [7]:
# загрузка нейросети rubert-tiny2
model = SentenceTransformer('cointegrated/rubert-tiny2', device=device)
model.max_seq_length = 128

# эмбеддинги валидации
val_honest_query_embeddings = model.encode(
    val_queries_honest['query_text'].tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=False
)

import os
if os.path.exists('corpus_embeddings.npy'):
    corpus_embeddings = np.load('corpus_embeddings.npy')
else:
    corpus_embeddings = model.encode(
        train_corpus_df['corpus_text'].tolist(),
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    np.save('corpus_embeddings.npy', corpus_embeddings)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 14388.24it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# обучение tf-idf и расчет recall@50 на валидации
val_vectorizer = TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2))
X_c_val = val_vectorizer.fit_transform(train_corpus_df['corpus_text'])
X_q_val = val_vectorizer.transform(val_queries_honest['query_text'])

corpus_cats_v = train_corpus_df['item_category_id'].values
corpus_locs_v = train_corpus_df['item_location_id'].values
corpus_ids_v = train_corpus_df['item_id'].values
query_cats_v = val_queries_honest['search_category'].values
query_locs_v = val_queries_honest['search_location_id'].values
query_uids_v = val_queries_honest['query_uid'].values

val_preds = {}
batch_size = 100
total_v = len(val_queries_honest)

for start_idx in range(0, total_v, batch_size):
    end_idx = min(start_idx + batch_size, total_v)
    s_t = (X_q_val[start_idx:end_idx] @ X_c_val.T).toarray()
    s_d = val_honest_query_embeddings[start_idx:end_idx] @ corpus_embeddings.T
    
    b_c = query_cats_v[start_idx:end_idx, None]
    b_l = query_locs_v[start_idx:end_idx, None]
    
    cat_m = (b_c == 0) | (b_c == corpus_cats_v[None, :])
    loc_m = (b_l == corpus_locs_v[None, :])
    mult = np.where(cat_m, np.where(loc_m, 3.2, 1.0), 0.001)
    
    s_t_b = s_t * mult
    s_d_b = s_d * mult
    
    max_t = np.maximum(s_t_b.max(axis=1, keepdims=True), 1e-6)
    max_d = np.maximum(s_d_b.max(axis=1, keepdims=True), 1e-6)
    s_hyb = 0.40 * (s_t_b / max_t) + 0.60 * (s_d_b / max_d)
    
    top50 = np.argpartition(-s_hyb, 50, axis=1)[:, :50]
    for i, q_idx in enumerate(range(start_idx, end_idx)):
        val_preds[query_uids_v[q_idx]] = corpus_ids_v[top50[i]]

val_score = compute_recall_at_k(val_preds, val_ground_truth_honest, k=50)
print(f'recall@50 на валидации: {val_score:.4f}')

recall@50 на валидации: 0.6477


## 2. Финальный инференс на тестовом бенчмарке

In [9]:
# загрузка тестовых файлов
benchmark_queries = pd.read_parquet('dataset/benchmark_queries.parquet')
benchmark_items = pd.read_parquet('dataset/benchmark_items.parquet')

# подготовка текстов тестового корпуса
pl_bi = pl.from_pandas(benchmark_items)
bt = pl_bi['item_title_raw'].fill_null('') + ' '
bp = pl_bi['item_infm_params_text'].fill_null('') + ' '
bd = pl_bi['item_description_raw'].fill_null('')

benchmark_items['corpus_text'] = (
    (bt + bt + bt + bp + bp + bd)
    .str.to_lowercase()
    .str.replace_all('ё', 'е')
    .to_pandas()
)

pl_bq = pl.from_pandas(benchmark_queries)
bq = pl_bq['search_query'].fill_null('') + ' '
bqp = pl_bq['search_infm_params_text'].fill_null('')

benchmark_queries['query_text'] = (
    (bq + bqp)
    .str.to_lowercase()
    .str.replace_all('ё', 'е')
    .to_pandas()
)

In [10]:
# обучение tf-idf на тестовом корпусе
bench_vectorizer = TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2))
X_bench_corpus = bench_vectorizer.fit_transform(benchmark_items['corpus_text'])
X_bench_queries = bench_vectorizer.transform(benchmark_queries['query_text'])
print(f'признаков в тестовом словаре: {X_bench_corpus.shape[1]}')

признаков в тестовом словаре: 2785768


In [11]:
# кодирование тестовых запросов и товаров нейросетью
if os.path.exists('bench_query_embeddings.npy') and os.path.exists('bench_corpus_embeddings.npy'):
    print('загрузка сохраненных тестовых эмбеддингов с диска')
    bench_query_embeddings = np.load('bench_query_embeddings.npy')
    bench_corpus_embeddings = np.load('bench_corpus_embeddings.npy')
else:
    bench_query_embeddings = model.encode(
        benchmark_queries['query_text'].tolist(),
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    bench_corpus_embeddings = model.encode(
        benchmark_items['corpus_text'].tolist(),
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    np.save('bench_query_embeddings.npy', bench_query_embeddings)
    np.save('bench_corpus_embeddings.npy', bench_corpus_embeddings)

загрузка сохраненных тестовых эмбеддингов с диска


In [12]:
# финальный гибридный поиск
bench_corpus_cats = benchmark_items['item_category_id'].values
bench_corpus_locs = benchmark_items['item_location_id'].values
bench_corpus_ids = benchmark_items['item_id'].values
bench_query_cats = benchmark_queries['search_category'].values
bench_query_locs = benchmark_queries['search_location_id'].values
bench_query_ids = benchmark_queries['query_id'].values

total_bench = len(benchmark_queries)
predictions_final = []

for start_idx in range(0, total_bench, batch_size):
    end_idx = min(start_idx + batch_size, total_bench)
    s_t = (X_bench_queries[start_idx:end_idx] @ X_bench_corpus.T).toarray()
    s_d = bench_query_embeddings[start_idx:end_idx] @ bench_corpus_embeddings.T
    
    b_c = bench_query_cats[start_idx:end_idx, None]
    b_l = bench_query_locs[start_idx:end_idx, None]
    
    cat_m = (b_c == 0) | (b_c == bench_corpus_cats[None, :])
    loc_m = (b_l == bench_corpus_locs[None, :])
    mult = np.where(cat_m, np.where(loc_m, 3.2, 1.0), 0.001)
    
    s_t_b = s_t * mult
    s_d_b = s_d * mult
    
    max_t = np.maximum(s_t_b.max(axis=1, keepdims=True), 1e-6)
    max_d = np.maximum(s_d_b.max(axis=1, keepdims=True), 1e-6)
    s_hyb = 0.40 * (s_t_b / max_t) + 0.60 * (s_d_b / max_d)
    
    top50 = np.argpartition(-s_hyb, 50, axis=1)[:, :50]
    for i in range(end_idx - start_idx):
        predictions_final.append(' '.join(bench_corpus_ids[top50[i]]))

# сохранение answer.csv
answer_df = pd.DataFrame({
    'query_id': bench_query_ids,
    'answer': predictions_final
})

assert len(answer_df) == 2452
assert all(len(ans.split(' ')) == 50 for ans in answer_df['answer'])
answer_df.to_csv('answer.csv', index=False)
print('файл answer.csv успешно сгенерирован')
print(answer_df.head(2))

файл answer.csv успешно сгенерирован
           query_id                                             answer
0  70DfDUpwjxB4lzFd  6042fecdd7753112 b92ee8f432cec2d1 603623b4bd9e...
1  JTrdTaZJvSiLPkXj  cbeccbecb1fb8d86 16ea26ff0aa0450a 8703b2a0442c...
